# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the [FAIR² croissant dataset on mass spectrometry analysis of extracellular vesicles](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, ensuring interoperable, machine-actionable metadata and easy programmatic access to structured data and documentation.

In [ ]:
# Install `mlcroissant` if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will provide an entry point for understanding which record sets are available and their associated metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)
print("Version:", metadata.version)
print("Published Date:", metadata.date_published)
print("License:", metadata.license)
print("Authors:")
for author in getattr(metadata, 'author', []):
    print("  -", getattr(author, 'name', author))
print("\nKeywords: ", getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available **record sets** in the dataset, along with their field `@id`s. 

Record sets are data tables/objects typically containing the bulk of tabular data. You should always refer to them, and their fields, **by their `@id`** for programmatic clarity and reproducibility.

In [ ]:
# List all available RecordSets and their fields by @id

record_set_objs = getattr(metadata, 'record_set', [])

if not record_set_objs:
    print("No record sets found in the dataset metadata. \nPlease make sure the schema contains 'record_set'.")
else:
    for rec in record_set_objs:
        print(f'RecordSet: @id={rec["@id"]}')
        print(f'  Name: {rec.get("name", "(No name)")}',)
        print('  Fields:')
        for field in rec.get('field', []):
            # field can be a str @id or a dict, depending on how the library loads it
            field_id = field["@id"] if isinstance(field, dict) and "@id" in field else field
            print(f'    - {field_id}')
        print('---')

# If you know a main table/recordSet, you can explore a few records:
# (For demonstration, replace with the real @id string below:)
main_recordset_id = None
if record_set_objs:
    main_recordset_id = record_set_objs[0]["@id"]
    print(f'First RecordSet @id: {main_recordset_id}')
    # Print a few sample records
    print("Record samples:")
    for i, rec in enumerate(dataset.records(record_set=main_recordset_id)):
        print(rec)
        if i >= 2: break

## 3. Data Extraction
Load data into a pandas DataFrame from a chosen record set. Use the **record set `@id` and field `@id`s** from above to extract and inspect records programmatically.

In [ ]:
# Build a list of all record set @ids
record_set_ids = [rec["@id"] for rec in getattr(metadata, 'record_set', [])]

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded {len(df)} records for RecordSet {rec_id}")

# Pick the primary table for demonstration (replace with actual @id if known)
if record_set_ids:
    main_rec_id = record_set_ids[0]
    print(f"Columns of main record set ({main_rec_id}):")
    print(dataframes[main_rec_id].columns.tolist())
    display(dataframes[main_rec_id].head())

## 4. Exploratory Data Analysis (EDA)
Now apply common data processing steps: filter records, transform numeric fields, group, clean, and prepare for analysis.

We'll:
- Select a numeric field (e.g., coverage, molecular_weight, or abundance)
- Filter based on a threshold
- Normalize values
- Optionally group by a categorical field

**Please check field names using the previous steps. Use the field `@id` (column name) in all analyses.**

In [ ]:
# Choose record set and field @ids for demonstration (update as needed based on your printouts above)

# Example placeholder ids (change these to actual ones from your dataset)
# numeric_field_id = '@id_of_numeric_field' # e.g., 'cr:field:mw'
# group_field_id = '@id_of_group_field'     # e.g., 'cr:field:sample_id'

main_rec_id = record_set_ids[0] if record_set_ids else None
if main_rec_id is not None:
    df = dataframes[main_rec_id]
    print('Field columns:', df.columns.tolist())
    # Select a likely numeric field by guessing
    sample_numeric_fields = [c for c in df.columns if c.lower() in ('mw', 'molecular_weight', 'abundance', 'coverage', 'peptides', 'total_peptides')]
    if sample_numeric_fields:
        numeric_field = sample_numeric_fields[0]
    else:
        # Fallback on first available float/integer
        numeric_field = df.select_dtypes(include=['float', 'int']).columns.tolist()[0] if df.select_dtypes(include=['float', 'int']).shape[1] else df.columns[0]

    # Filter entries with numeric_field > threshold (adjust threshold based on your field)
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else None
    if threshold is not None:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (first 5 rows):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by first categorical field if available
        sample_cat_fields = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field]
        if sample_cat_fields:
            group_field = sample_cat_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped average {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print(f"Cannot filter: '{numeric_field}' does not seem to be numeric in this dataset.")

## 5. Visualization
Let's visualize the distribution of numeric values and compare across groups (if available).

We will use matplotlib and seaborn for plotting. Adjust `numeric_field` and `group_field` variables as appropriate for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rec_id is not None and 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # Boxplot by group (if available)
    if 'group_field' in locals() and group_field in filtered_df.columns:
        plt.figure(figsize=(10,5))
        # Avoid overplotting if high cardinality
        group_limited = filtered_df[group_field].astype(str)
        if group_limited.nunique() > 20:
            # Use only the top 20 groups
            top_groups = group_limited.value_counts().index[:20]
            filtered_df_box = filtered_df[filtered_df[group_field].isin(top_groups)]
        else:
            filtered_df_box = filtered_df
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df_box)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print('No filtered or numeric data to plot. Please check previous EDA cell for filtered results.')

## 6. Conclusion

- We have demonstrated how to programmatically explore and process a FAIR² Croissant dataset representing mass spectrometry results for extracellular vesicles.
- Using `mlcroissant`, we extracted the dataset structure, loaded records using *record set `@id`s*, and explored numeric and categorical fields.
- Initial filtering, normalization, and group analyses were performed, followed by basic visualizations of data distributions.

### Next steps:
- Investigate additional record sets for metadata or experimental details
- Construct more targeted queries or modeling using specific protein 'accession', peptide, or abundance fields
- Join across record sets (by entity or sample identifiers) if dataset schema allows
- Refer to original Croissant schema at https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json for details on entity relationships and semantics

*Always use field and record set `@id`s programmatically to ensure reproducible, interpretable access!*